# Token Binding Analysis

Analyze how tokens become bound to contextual roles: binding trajectory,
strength comparison, source attribution, competition, and summary.

## Why This Matters

Token binding tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_binding_analysis import (
    token_role_binding, binding_strength_comparison,
    binding_source_attribution, binding_competition,
    token_binding_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Token Role Binding

How much does a token's representation diverge from its static embedding?

In [ ]:
result = token_role_binding(model, tokens, position=-1)
print(f"Strongly bound: {result['is_strongly_bound']}\n")
for p in result['per_layer']:
    bar = '█' * int(max(0, (1 - p['cosine_to_embed']) * 20))
    print(f"  Layer {p['layer']}: cos_embed={p['cosine_to_embed']:.4f}, "
          f"div={p['divergence']:.4f} {bar}")

## Binding Strength Comparison

Which positions get the most contextual modification?

In [ ]:
result = binding_strength_comparison(model, tokens)
print(f"Most bound: pos {result['most_bound_position']}")
print(f"Least bound: pos {result['least_bound_position']}\n")
for p in result['per_position']:
    bar = '█' * int(min(p['divergence'] * 10, 30))
    print(f"  Pos {p['position']} (tok {p['token_id']:3d}): "
          f"cos={p['cosine_to_embed']:.4f}, div={p['divergence']:.4f} {bar}")

## Binding Source Attribution

Is attention or MLP responsible for contextual binding?

In [ ]:
result = binding_source_attribution(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn_perp={p['attn_perpendicular']:.4f}, "
          f"mlp_perp={p['mlp_perpendicular']:.4f} [{p['binding_source']}]")

## Binding Competition

Which token embeddings compete for a position's representation?

In [ ]:
result = binding_competition(model, tokens, position=-1, top_k=5)
print(f"Original token: {result['original_token']}\n")
for p in result['per_layer']:
    comps = ', '.join(f'tok{c["token_id"]}({c["cosine"]:.3f})' for c in p['competitors'][:3])
    print(f"  Layer {p['layer']}: top={p['top_competitor']} ({p['top_cosine']:.4f}) | {comps}")

## Binding Summary

Cross-position overview of token binding patterns.

In [ ]:
result = token_binding_summary(model, tokens)
print(f"Mean binding rate: {result['mean_binding_rate']:.4f}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']} (tok {p['token_id']:3d}): "
          f"init_cos={p['initial_cosine']:.4f}, final_cos={p['final_cosine']:.4f}, "
          f"rate={p['binding_rate']:.4f}")